In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

import numpy as np

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)  # гарантируем воспроизводимость

run_env = os.getenv('RUN_ENV', '')
if run_env == 'COLLAB':
  from google.colab import drive
  ROOT_DIR = '/content/drive'
  drive.mount(ROOT_DIR)
  print('Google drive connected')
  DRIVE_DATA_DIR = 'ml_course_data'
  root_data_dir = os.path.join(ROOT_DIR, 'MyDrive', DRIVE_DATA_DIR)
  sys.path.append(os.path.join(ROOT_DIR, 'MyDrive', 'src'))
else:
  root_data_dir = os.getenv('DATA_DIR', '/srv/data')

if not os.path.exists(root_data_dir):
  raise RuntimeError('Отсутствует директория с данными')
else:
  print('Содержимое директории %s: %s' % (root_data_dir, os.listdir(root_data_dir)[:10]))

Содержимое директории /Users/adzhumurat/PycharmProjects/home_brain/data: ['interview_preparation', 'agentic_specs', 'cracking_llm_basics.md', 'Small Language Models for Efficient Agentic Tool Calling, Outperforming Large Models with Targeted Fine-tuning.md', 'qa_knowledgebase', 'knowledgebase.jsonl', 'recognized_speech', 'linkedin_posts', 'youtube', 'it_ops.md']


In [2]:
from mindbase_layer.retrieve_md import DocumentIndex

# md_path = '/Users/adzhumurat/PycharmProjects/ai_product_engineer/slides/cracking_llm/cracking_llm_basics.md'
md_path = '/Users/adzhumurat/PycharmProjects/ai_product_engineer/src/assistant/.analysis/skeleton.md'

doc_index = DocumentIndex.from_md_file(md_path)
doc_index.iloc[1]

DocumentNode(header='# tools.py', body='**Imports**\n- `from sklearn.feature_extraction.text import TfidfVectorizer`\n\n**Classes**\n- DocumentIndex\n\n**Functions**\n- read_md_nodes(file_path) -> list[DocumentNode] | Read a markdown file and return a list of DocumentNode, one per header section.', source=PosixPath('/Users/adzhumurat/PycharmProjects/ai_product_engineer/src/assistant/.analysis/skeleton.md'), parent=None, node_name='8ed1bbdcb2aff942efe0a8f040314dfd')

In [3]:
print(doc_index.iloc[1].body)

**Imports**
- `from sklearn.feature_extraction.text import TfidfVectorizer`

**Classes**
- DocumentIndex

**Functions**
- read_md_nodes(file_path) -> list[DocumentNode] | Read a markdown file and return a list of DocumentNode, one per header section.


In [4]:
# for i in doc_index.get_childs('84f90a99abefa63476537580c7b83647'):
#     print(i)
#     print('=' * 10)

# from mindbase_layer.formatting import print_search_results  

# results = doc_index.search('Byte pair encodings')

# for score, r in results:
#     print(r)
#     print('='*10)
#     print()

# Doc similarity

In [5]:
jobs_desription_path = '~/Downloads/nebius_jobs_md'

dir_index = DocumentIndex.from_dir(jobs_desription_path)
dir_index.describe()

'blocks: 4599, chapters: 4599, chars: 1503080, avg chars/block: 381'

In [26]:
from pathlib import Path

import pandas as pd


path_obj = Path(jobs_desription_path).expanduser()

files = [f.name for f in path_obj.iterdir() if f.is_file()]
df = pd.DataFrame(files, columns=['file_path'])
df.to_csv('~/Downloads/vacansies.csv', index=False)
df.head()

,file_path
0,4554517101_field-network-engineer.md
1,4816613101_senior-frontend-developer.md
2,4797070101_data-center-it-technician.md
3,4837329101_hr-operations-specialist.md
4,4811659101_senior-data-business-analyst.md


In [7]:
resume_path = '~/Downloads/aleksandr_dzhumurat_ai_generalist.md'

resume_md = DocumentIndex.from_md_file(resume_path)

for score, doc in dir_index.search(query=resume_md.to_md()):
    print(f'{score:.4f}')
    print(doc)

0.3152
## Must-haves:
- 3+ years of experience in data engineering or related roles. ·
- Experience building and maintaining data pipelines using orchestration tools (e.g., Airflow, Prefect, Dagster). ·
- Strong proficiency in SQL and solid programming skills in Python. ·
- Experience with distributed data processing fra...
7651f0e3131c85fbe7c38a309af33f0b
0.3027
## The role
The Data Engineering team is responsible for building and maintaining robust data infrastructure that powers analytics, and business intelligence across Nebius. We design and implement scalable data pipelines, optimize data storage and processing, and enable data-driven decision making across the or...
64564173cd2d5eaca81c371f73a863e4
0.2995
## We expect you to have:
- Experience as a data scientist or applied ML practitioner (5+ years). ·
- Experience building and deploying LLM-based systems in production. ·
- Experience working in production environments with model monitoring and iteration. ·
- Experience with mo

In [21]:
import pandas as pd

df = pd.DataFrame([
    (score, doc.source)
    for score, doc
    in dir_index.search(query=resume_md.to_md(), top_k=4500)
], columns = ['score', 'doc'])

df.head()

,score,doc
0,0.315200,/Users/adzhumurat/Downloads/nebius_jobs_md/472...
1,0.302651,/Users/adzhumurat/Downloads/nebius_jobs_md/472...
2,0.299474,/Users/adzhumurat/Downloads/nebius_jobs_md/481...
3,0.286866,/Users/adzhumurat/Downloads/nebius_jobs_md/482...
4,0.257372,/Users/adzhumurat/Downloads/nebius_jobs_md/481...


In [22]:
df.iloc[-1]['doc'], df.iloc[-1]['score']

(PosixPath('/Users/adzhumurat/Downloads/nebius_jobs_md/4705452101_construction-project-manager-data-centers.md'),
 np.float64(0.006054609093639718))

In [ ]:
buffer = ''
for f in pd.read_csv('~/Downloads/relevant_jobs.csv').rename(columns={'File_name': 'file_name'}).to_dict(orient='records'):
    file_path = os.path.join(jobs_desription_path, f['file_name'])
    print(file_path)
    with open(os.path.expanduser(file_path), 'r') as ff:
        buffer += f'{buffer}\n\n{ff.read()}'
with open('~/Downloads/raw_jobs.md', 'w') as f:
    f.write(buffer)

~/Downloads/nebius_jobs_md/4773256101_partner-solutions-architect.md
~/Downloads/nebius_jobs_md/4824821101_senior-software-engineer-agentic-search-runtime.md
~/Downloads/nebius_jobs_md/4753426101_technical-project-manager-turnups-and-setup-of-operations.md
~/Downloads/nebius_jobs_md/4845016101_developer-advocate-token-factory.md
~/Downloads/nebius_jobs_md/4837235101_technical-program-manager.md
~/Downloads/nebius_jobs_md/4844282101_senior-technical-project-manager-base-infra.md
~/Downloads/nebius_jobs_md/4639672101_lead-cloud-solutions-architect.md
~/Downloads/nebius_jobs_md/4648728101_senior-ml-solutions-architect-token-factory.md
~/Downloads/nebius_jobs_md/4817418101_sr-solution-engineer-architect-gpu-infrastructure.md
~/Downloads/nebius_jobs_md/4754617101_senior-ml-engineer-token-factory.md
~/Downloads/nebius_jobs_md/4705105101_senior-technical-project-manager-devtools-observability.md
~/Downloads/nebius_jobs_md/4854260101_automations-engineer.md
~/Downloads/nebius_jobs_md/477610810